# Introduction:

*Stuff to add*

- Contents structure
- Explain what the project is about
- How we carried out the analysis
- What are the tools used



## Data Acquisition:

### Primary: Guardian API
We will collect articles from the Guardian API using commodity-related search terms (e.g. oil, natural gas, copper, wheat, OPEC, energy crisis) filtered by relevant sections (e.g. Business, Environment, World News). For each article, we may extract the publication date, headline, section, tags, word count, and body text. We plan to collect data spanning approximately 2014 - 2026 to capture multiple geopolitical cycles.

### Secondary: Commodity Price Data
To contextualise media coverage against market movements, we will source historical commodity price data from Yahoo Finance (via the yfinance Python library) or publicly available datasets. This will include daily/weekly prices for key commodities such as Brent Crude, natural gas, gold, copper, and wheat.

- Query the Guardian API with commodity-related keywords and display results across the full date range

- Download commodity price time series from Yahoo Finance using yfinance

- Store raw data in structured formats (JSON/CSV) for reproducibility



In [1]:
import json
import requests
import pandas as pd
import time



with open('keys.json') as f:
    key = json.load(f)

API_KEY = key['guardian']['api_key']
BASE_URL = 'https://content.guardianapis.com'

# generate (first_day, last_day) pairs for each month
month_starts = pd.date_range("2020-01-01", "2026-03-31", freq="MS")
date_ranges = [
    (d.strftime("%Y-%m-%d"), (d + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d"))
    for d in month_starts
]

ARTICLES_PER_MONTH = 50
all_results = []



for from_date, to_date in date_ranges:
    for page in range(1, (ARTICLES_PER_MONTH // 50) + 1):
        parameters = {
            "api-key": API_KEY,
            "q": "oil OR natural gas OR gold OR OPEC OR energy crisis",
            "page-size": ARTICLES_PER_MONTH,
            "page": page,
            "show-fields": "bodyText",
            "from-date": from_date,
            "to-date": to_date,
            "order-by": "relevance"
        }
        response = requests.get(f"{BASE_URL}/search", params=parameters)

        data = response.json()['response']

        for article in data['results']:
            all_results.append({
                'date': article.get('webPublicationDate'),
                'section': article.get('sectionName'),
                'title': article.get('webTitle'),
            })

        if page >= data['pages']:
            break
        time.sleep(0.5)


print(f"Done.  Total: {len(all_results)}")



/Users/pramod/Documents/GitHub/st115/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


KeyError: 'response'

In [2]:
import yfinance as yf


# Define commodity tickers
# Yahoo Finance uses these symbols for common commodities:
# all futures
commodities = {
    "Gold":        "GC=F",   
    "Brent Oil":   "BZ=F",   # brent crude oil
    "Natural Gas": "NG=F",   
    "Wheat":       "ZW=F",   
}

# Download historical data
data = yf.download(
    tickers=list(commodities.values()),
    start="2020-01-01",
    end="2025-12-31",
    interval="1wk"       # 1d for daily, 1wk for weekly, 1mo for monthly
)

# Extract closing prices only
close_prices = data["Close"]
close_prices.columns = list(commodities.keys())

df_commodities_prices = pd.DataFrame(close_prices)


print(close_prices.head(50))


[*********************100%***********************]  4 of 4 completed

                 Gold    Brent Oil  Natural Gas   Wheat
Date                                                   
2020-01-01  68.269997  1571.800049        2.162  550.25
2020-01-08  64.489998  1542.400024        2.187  568.50
2020-01-15  64.589996  1556.400024        1.895  581.50
2020-01-22  59.509998  1569.199951        1.934  569.75
2020-01-29  53.959999  1550.400024        1.872  557.25
2020-02-05  54.009998  1565.599976        1.788  542.00
2020-02-12  57.750000  1600.000000        1.981  566.75
2020-02-19  54.950001  1646.900024        1.847  539.00
2020-02-26  51.860001  1642.099976        1.800  528.75
2020-03-04  37.220001  1659.099976        1.936  526.75
2020-03-11  28.730000  1524.900024        1.729  499.25
2020-03-18  27.150000  1660.199951        1.653  561.50
2020-03-25  22.740000  1583.400024        1.640  568.75
2020-04-01  31.870001  1664.800049        1.852  549.25
2020-04-08  29.600000  1756.699951        1.650  548.75
2020-04-15  19.330000  1678.199951        1.821 

# IDA

In [ ]:
# mantra 

# Data wrangling

# EDA  

# Visualisation

rian